# PolyAtlas — Notebook 02c: Harvey 2022 Ingestion

**Project:** Cross-Campaign Polyreactivity Atlas  
**Notebook:** `PolyAtlas_02c_harvey_ingestion`  
**Version:** `v0.1.0`  
**Date:** 2026-04-19

## Purpose
Ingest Harvey et al. 2022 nanobody polyreactivity data as a third independent assay source, preparing it for use as a held-out test set in the J/JS/JSB cross-assay experiment (Notebook 03c).

## Data source
GitHub: `debbiemarkslab/nanobody-polyreactivity` (Marks lab, Harvard). Uploaded by user to Drive at `/content/drive/MyDrive/PolyAtlas/raw_data/harvey_2022/nanobody-polyreactivity-main/backend/app/experiments/`

Three CSVs:
- `high_polyreactivity_high_throughput.csv` (71,772 sequences) — FACS-sorted HIGH polyreactivity pool, labeled 1
- `low_polyreactivity_high_throughput.csv` (69,702 sequences) — FACS-sorted LOW polyreactivity pool, labeled 0
- `low_throughput_polyspecificity_scores_w_exp.csv` (48 sequences) — indexed panel with continuous PSR scores (mean of 3 biological replicates, range ~1.8 to ~99)

## Sampling strategy
From the two big pool files, sample **1,000 each** (balanced, seeded) to get ~2,000 binary-labeled nanobodies. Plus all 48 from the low-throughput panel for continuous-label cross-validation. Total target: **~2,048 nanobodies**.

We don't need all 141k sequences — diminishing returns for our scale, plus AbLang embedding of 141k sequences would take hours on CPU and nontrivial time on GPU.

## Key advantages over Boughter
Harvey data is already fully preprocessed:
- ANARCI annotation done (IMGT numbering in position columns 1-128)
- CDR sequences pre-extracted (`CDR1_nogaps`, `CDR2_nogaps`, `CDR3_nogaps`)
- Biophysical features precomputed (`CDRS_IP`, `CDRS_HP`, `CDR1/2/3_length`, glycosylation flags)
- V-gene and V-identity fields present (mostly NaN because synthetic library, but that's fine)

**We skip ANARCI entirely.** We only reconstruct the full VH sequence from position columns (needed for AbLang) and run AbLang embedding.

## Important caveats
- **All Harvey data is nanobody (VHH).** No conventional heavy+light antibody pairing. Single-domain Ig fold with different surface architecture than human IgG.
- **AbLang was trained on conventional antibodies.** Embeddings will be lower quality than for matched-domain data. We accept this as a form of domain shift that we'll handle at analysis time.
- **The Harvey PSR reagent is Sf9 insect cell lysate** — similar *concept* to Adimab PSR (mixed cellular proteins) but different reagent and different detection system (FACS vs flow cytometry).
- **Species mostly alpaca.** ANARCI column says 'alpaca' for 99.99% of sequences. We'll map this to our species schema as 'alpaca' (already in our allowed_species list).

## Inputs
- `/content/drive/MyDrive/PolyAtlas/raw_data/harvey_2022/nanobody-polyreactivity-main/backend/app/experiments/high_polyreactivity_high_throughput.csv`
- `/content/drive/MyDrive/PolyAtlas/raw_data/harvey_2022/nanobody-polyreactivity-main/backend/app/experiments/low_polyreactivity_high_throughput.csv`
- `/content/drive/MyDrive/PolyAtlas/raw_data/harvey_2022/nanobody-polyreactivity-main/backend/app/experiments/low_throughput_polyspecificity_scores_w_exp.csv`
- `PolyAtlas_02b_external_validation_v0.1.0/unified_master_*.pkl` + embeddings (to extend)

## Outputs
- `PolyAtlas_02c_harvey_ingestion_v0.1.0/`
  - `harvey_processed_v0.1.0.pkl` — Harvey data in unified master format
  - `harvey_embeddings_v0.1.0.npy` — AbLang 768-dim embeddings for Harvey
  - `harvey_index_panel_v0.1.0.pkl` — the 48 indexed nanobodies with continuous PSR
  - `unified_master_with_harvey_v0.1.0.pkl` — master DF + Harvey (for use in 03c)
  - `unified_embeddings_with_harvey_v0.1.0.npy` — matching embeddings (aligned to master)
  - `ingestion_summary_v0.1.0.json` — audit log

## Expected runtime
~15-25 min total:
- CSV loading + sampling: ~2 min
- VH reconstruction: ~1 min
- AbLang embedding of ~2,050 sequences: ~10-20 min on A100
- Merge + save: ~1 min

## Version log
- **v0.1.0 (2026-04-19):** Initial build. Samples 1,000 high + 1,000 low from pool files + 48 indexed.

## §A. Click-alive

In [ ]:
from IPython.display import display, Javascript

display(Javascript('''
function ClickConnect(){
    console.log("keep-alive @ " + new Date().toLocaleTimeString());
    const selectors = ["#top-toolbar > colab-connect-button", "colab-connect-button", "#connect"];
    for (const sel of selectors) {
        const el = document.querySelector(sel);
        if (el) {
            if (el.shadowRoot) {
                const inner = el.shadowRoot.querySelector("#connect");
                if (inner) { inner.click(); return; }
            }
            el.click();
            return;
        }
    }
}
setInterval(ClickConnect, 60000);
'''))

print("✓ Click-alive injected.")

<IPython.core.display.Javascript object>

✓ Click-alive injected.


## §B. Force-mount Drive + paths

In [ ]:
from google.colab import drive
from pathlib import Path
import json

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/PolyAtlas')
NOTEBOOK_NAME = "PolyAtlas_02c_harvey_ingestion"
PROJECT_VERSION = "0.1.0"
DRIVE_OUTPUT = DRIVE_ROOT / f"{NOTEBOOK_NAME}_v{PROJECT_VERSION}"
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Harvey input files
HARVEY_DIR = DRIVE_ROOT / 'raw_data' / 'harvey_2022'
HARVEY_HIGH = HARVEY_DIR / 'high_polyreactivity_high_throughput.csv'
HARVEY_LOW  = HARVEY_DIR / 'low_polyreactivity_high_throughput.csv'
HARVEY_LT   = HARVEY_DIR / 'low_throughput_polyspecificity_scores_w_exp.csv'

# Existing unified master (from 02b) to extend
V02B_DIR = DRIVE_ROOT / 'PolyAtlas_02b_external_validation_v0.1.0'
UNIFIED_MASTER_PKL = V02B_DIR / 'PolyAtlas_02b_external_validation_unified_master_v0.1.0.pkl'
UNIFIED_EMB_NPY   = V02B_DIR / 'PolyAtlas_02b_external_validation_unified_embeddings_v0.1.0.npy'

required = {
    'Harvey HIGH CSV': HARVEY_HIGH,
    'Harvey LOW CSV': HARVEY_LOW,
    'Harvey low-throughput CSV': HARVEY_LT,
    'Unified master (02b)': UNIFIED_MASTER_PKL,
    'Unified embeddings (02b)': UNIFIED_EMB_NPY,
}

print(f"✓ Drive mounted. Output: {DRIVE_OUTPUT}\n")
missing = []
for name, path in required.items():
    status = '✓' if path.exists() else '✗'
    print(f"  {status} {name}: {path}")
    if not path.exists():
        missing.append(name)
if missing:
    raise FileNotFoundError(f"Missing inputs: {missing}")

Mounted at /content/drive
✓ Drive mounted. Output: /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0

  ✓ Harvey HIGH CSV: /content/drive/MyDrive/PolyAtlas/raw_data/harvey_2022/high_polyreactivity_high_throughput.csv
  ✓ Harvey LOW CSV: /content/drive/MyDrive/PolyAtlas/raw_data/harvey_2022/low_polyreactivity_high_throughput.csv
  ✓ Harvey low-throughput CSV: /content/drive/MyDrive/PolyAtlas/raw_data/harvey_2022/low_throughput_polyspecificity_scores_w_exp.csv
  ✓ Unified master (02b): /content/drive/MyDrive/PolyAtlas/PolyAtlas_02b_external_validation_v0.1.0/PolyAtlas_02b_external_validation_unified_master_v0.1.0.pkl
  ✓ Unified embeddings (02b): /content/drive/MyDrive/PolyAtlas/PolyAtlas_02b_external_validation_v0.1.0/PolyAtlas_02b_external_validation_unified_embeddings_v0.1.0.npy


## §1. Install and imports

Just AbLang for embedding. No ANARCI needed (Harvey data is already annotated).

In [ ]:
!pip install -q ablang pandas numpy

import pandas as pd
import numpy as np
import ablang
import torch
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"✓ Imports loaded")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")

✓ Imports loaded
✓ CUDA available: True
  Device: NVIDIA A100-SXM4-40GB


## §2. Load Harvey CSVs and sample

Sampling: 1,000 from HIGH + 1,000 from LOW + all 48 from indexed panel. Seeded random sample for reproducibility.

In [ ]:
# Load only the columns we need (CSVs are 59 MB each with 194 columns — we only want ~15)

# The position columns for VH reconstruction are between 'j_identity' and 'CDR1_withgaps'.
# First, read the header to figure out which columns we need.

with open(HARVEY_HIGH) as f:
    header_line = f.readline().strip().split(',')

# The header has an empty first field (row index) — we'll let pandas handle that
# Identify the position columns
try:
    j_idx = header_line.index('j_identity')
    cdr1_wg_idx = header_line.index('CDR1_withgaps')
    POSITION_COLS = header_line[j_idx + 1 : cdr1_wg_idx]
    print(f"Position columns (count): {len(POSITION_COLS)}")
    print(f"First: {POSITION_COLS[:5]}  Last: {POSITION_COLS[-5:]}")
except ValueError as e:
    print(f"Header parse error: {e}")
    raise

# Columns we want to load from each CSV
DESIRED_COLS = (
    ['Id', 'hmm_species', 'chain_type', 'v_gene', 'v_identity', 'j_gene']
    + POSITION_COLS
    + ['CDR1_nogaps', 'CDR2_nogaps', 'CDR3_nogaps', 'CDRS_nogaps',
       'CDR1_length', 'CDR2_length', 'CDR3_length',
       'CDRS_IP', 'CDRS_HP']
)
print(f"\nTotal columns to load per CSV: {len(DESIRED_COLS)}")

Position columns (count): 151
First: ['1', '2', '3', '4', '5']  Last: ['124', '125', '126', '127', '128']

Total columns to load per CSV: 166


In [ ]:
# Load HIGH and LOW files with only the desired columns
print("Loading Harvey HIGH polyreactivity file...")
hi_df = pd.read_csv(HARVEY_HIGH, usecols=lambda c: c in DESIRED_COLS, low_memory=False)
print(f"  Loaded: {hi_df.shape}")

print("Loading Harvey LOW polyreactivity file...")
lo_df = pd.read_csv(HARVEY_LOW, usecols=lambda c: c in DESIRED_COLS, low_memory=False)
print(f"  Loaded: {lo_df.shape}")

print("Loading Harvey indexed panel (low-throughput)...")
lt_df = pd.read_csv(HARVEY_LT, low_memory=False)
print(f"  Loaded: {lt_df.shape}")
print(f"  Columns of interest: Biorep average present: {'Biorep average' in lt_df.columns}")

Loading Harvey HIGH polyreactivity file...
  Loaded: (71772, 166)
Loading Harvey LOW polyreactivity file...
  Loaded: (69702, 160)
Loading Harvey indexed panel (low-throughput)...
  Loaded: (48, 193)
  Columns of interest: Biorep average present: True


In [ ]:
# Sample 1,000 from each of HIGH and LOW using seeded random sampling
SAMPLE_N = 1000

hi_sampled = hi_df.sample(n=SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
lo_sampled = lo_df.sample(n=SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)

hi_sampled['poly_label'] = 1  # HIGH pool → polyreactive
lo_sampled['poly_label'] = 0  # LOW pool → specific
hi_sampled['source_pool'] = 'high'
lo_sampled['source_pool'] = 'low'

print(f"Sampled HIGH: {len(hi_sampled)}  All labeled 1")
print(f"Sampled LOW:  {len(lo_sampled)}  All labeled 0")
print(f"\nSequence species check:")
print(f"  HIGH species: {hi_sampled['hmm_species'].value_counts().to_dict()}")
print(f"  LOW species:  {lo_sampled['hmm_species'].value_counts().to_dict()}")

harvey_ht_df = pd.concat([hi_sampled, lo_sampled], ignore_index=True)
print(f"\nCombined high-throughput sample: {len(harvey_ht_df)}")

Sampled HIGH: 1000  All labeled 1
Sampled LOW:  1000  All labeled 0

Sequence species check:
  HIGH species: {'alpaca': 1000}
  LOW species:  {'alpaca': 1000}

Combined high-throughput sample: 2000


## §3. Reconstruct full VH sequences

The IMGT-numbered position columns (1 through 128 with insertion codes) give us the VH sequence when joined and gaps removed.

In [ ]:
def reconstruct_vh(row, pos_cols):
    """Join position columns, drop gaps and NaNs, return full VH sequence."""
    aa_list = []
    for c in pos_cols:
        aa = row[c]
        if pd.notna(aa) and aa != '-':
            aa_list.append(str(aa).upper())
    return ''.join(aa_list)

# Apply to high-throughput sample
print("Reconstructing VH sequences for high-throughput sample...")
harvey_ht_df['VH_full'] = harvey_ht_df.apply(lambda r: reconstruct_vh(r, POSITION_COLS), axis=1)

# Apply to indexed panel (same column format)
# The low-throughput file has slightly different position column structure — let's check
# Its position columns also go between 'j_identity' and 'CDR1_withgaps'
lt_header = lt_df.columns.tolist()
try:
    lt_j_idx = lt_header.index('j_identity')
    lt_cdr1_idx = lt_header.index('CDR1_withgaps')
    LT_POSITION_COLS = lt_header[lt_j_idx + 1 : lt_cdr1_idx]
    print(f"Low-throughput position columns: {len(LT_POSITION_COLS)}")
    lt_df['VH_full'] = lt_df.apply(lambda r: reconstruct_vh(r, LT_POSITION_COLS), axis=1)
except ValueError:
    print("Low-throughput file has different schema; falling back to CDRS_nogaps usage.")
    lt_df['VH_full'] = None

# Check VH lengths
print(f"\nHT VH lengths: min={harvey_ht_df['VH_full'].str.len().min()}, "
      f"median={harvey_ht_df['VH_full'].str.len().median()}, "
      f"max={harvey_ht_df['VH_full'].str.len().max()}")

# Sanity check: CDR3 should be inside VH_full
mismatch = harvey_ht_df.apply(lambda r: r['CDR3_nogaps'] not in r['VH_full']
                               if pd.notna(r['CDR3_nogaps']) and pd.notna(r['VH_full']) else False, axis=1)
print(f"\nCDR3 not in VH (should be 0): {mismatch.sum()}")

# Sample verification
print("\nSample 3 reconstructed VH sequences:")
for i in [0, 500, 1500]:
    r = harvey_ht_df.iloc[i]
    print(f"  [{i}] label={r['poly_label']}, VH length={len(r['VH_full'])}, CDR3={r['CDR3_nogaps']}")

Reconstructing VH sequences for high-throughput sample...
Low-throughput position columns: 138

HT VH lengths: min=107, median=120.0, max=131

CDR3 not in VH (should be 0): 2

Sample 3 reconstructed VH sequences:
  [0] label=1, VH length=116, CDR3=AAFGWGYGY
  [500] label=1, VH length=120, CDR3=NKGYPLYDTGYYY
  [1500] label=0, VH length=121, CDR3=AVSQSWYWYYASDY


In [ ]:
# Check for any weird characters (like X, *, etc.) in the reconstructed sequences
STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWY')

def non_standard_chars(seq):
    return set(seq) - STANDARD_AA

harvey_ht_df['non_std'] = harvey_ht_df['VH_full'].apply(non_standard_chars)
has_weird = harvey_ht_df['non_std'].apply(lambda s: len(s) > 0)

print(f"Sequences with non-standard characters: {has_weird.sum()} / {len(harvey_ht_df)}")
if has_weird.sum() > 0:
    all_weird_chars = set()
    for s in harvey_ht_df.loc[has_weird, 'non_std']:
        all_weird_chars.update(s)
    print(f"  All non-standard chars found: {all_weird_chars}")

# Drop rows with non-standard characters if few; mask them if many
if has_weird.sum() > 0 and has_weird.sum() < 50:
    harvey_ht_df = harvey_ht_df[~has_weird].reset_index(drop=True)
    print(f"  Dropped {has_weird.sum()} rows. Remaining: {len(harvey_ht_df)}")
elif has_weird.sum() >= 50:
    # Mask non-standard with AbLang mask token '*'
    def mask_non_std(seq):
        return ''.join(aa if aa in STANDARD_AA else '*' for aa in seq)
    harvey_ht_df.loc[has_weird, 'VH_full'] = harvey_ht_df.loc[has_weird, 'VH_full'].apply(mask_non_std)
    print(f"  Masked non-standard residues in {has_weird.sum()} rows.")

harvey_ht_df = harvey_ht_df.drop(columns=['non_std'])

# Same check for low-throughput
lt_df['non_std'] = lt_df['VH_full'].apply(lambda s: non_standard_chars(s) if pd.notna(s) else set())
lt_weird = lt_df['non_std'].apply(lambda s: len(s) > 0)
print(f"\nLow-throughput sequences with non-standard: {lt_weird.sum()} / {len(lt_df)}")
lt_df = lt_df.drop(columns=['non_std'])

Sequences with non-standard characters: 0 / 2000

Low-throughput sequences with non-standard: 0 / 48


## §4. Build unified master rows for Harvey

Map Harvey columns to our unified schema. This must match the existing unified_master_df structure from Notebook 02b.

In [ ]:
# Load existing master to get expected schema
master_df_existing = pd.read_pickle(UNIFIED_MASTER_PKL)
print(f"Existing master shape: {master_df_existing.shape}")
print(f"Existing master columns: {list(master_df_existing.columns)}")
print(f"\nExisting sources: {master_df_existing['source'].value_counts().to_dict()}")

Existing master shape: (7700, 38)
Existing master columns: ['vh_seq', 'vl_seq', 'name', 'target', 'polyreactivity_psr', 'polyreactivity_bvp', 'polyreactivity_label', 'polyreactivity_score_source', 'tm_fab', 'hic_rt', 'smac_rt', 'ac_sins_dlambda', 'hek_titer', 'cic_rt', 'clinical_status', 'source', 'v_gene_external', 'id', 'species_anarci', 'chain_type', 'bitscore', 'CDR-H1', 'CDR-H2', 'CDR-H3', 'cdr_h3_len', 'v_call', 'v_identity', 'v_species', 'v_family', 'j_call', 'j_identity', 'j_species', 'polyreactivity_numreact', 'boughter_subset', 'boughter_frame_used', 'b_cell_subset', 'vh_germline_shehata', 'vl_germline_shehata']

Existing sources: {'cov_abdab': 3000, 'oas_naive': 2000, 'thera_sabdab': 994, 'boughter_2020_mouse': 481, 'shehata_2019': 400, 'boughter_2020_flu': 379, 'jain_2017': 137, 'boughter_2020_nat_hiv': 134, 'boughter_2020_gut_hiv': 75, 'boughter_2020_nat_cntrl': 50, 'boughter_2020_plos_hiv': 50}


In [ ]:
# Build Harvey HIGH-THROUGHPUT rows in unified format
def build_harvey_ht_rows(df):
    rows = []
    for i, r in df.iterrows():
        rows.append({
            'source': 'harvey_2022',
            'ab_id': f"harvey_{r['source_pool']}_{r['Id']}",
            'species': 'alpaca' if pd.isna(r.get('hmm_species')) else str(r.get('hmm_species')).lower(),
            'chain_type': 'H',  # VHH — heavy chain only
            'VH': r['VH_full'],
            'VL': None,  # nanobody — no light chain
            'CDR-H1': r['CDR1_nogaps'],
            'CDR-H2': r['CDR2_nogaps'],
            'CDR-H3': r['CDR3_nogaps'],
            'v_gene': r.get('v_gene') if pd.notna(r.get('v_gene')) else None,
            'v_identity': float(r['v_identity']) if pd.notna(r.get('v_identity')) else None,
            'v_family': None,  # synthetic library, no meaningful V family
            'j_gene': r.get('j_gene') if pd.notna(r.get('j_gene')) else None,
            'isotype': None,
            'format': 'VHH',
            # Labels
            'polyreactive_binary': int(r['poly_label']),  # 1 = high poly, 0 = low poly
            'polyreactive_continuous': None,  # HT data has no continuous score
            'assay_type': 'FACS_sort_PSR',
            'antigen_context': 'Sf9_insect_cell_lysate',
            'assay_family': 'harvey_PSR',
            # Subset metadata
            'subset_label': f"harvey_ht_{r['source_pool']}",
        })
    return pd.DataFrame(rows)

harvey_ht_master = build_harvey_ht_rows(harvey_ht_df)
print(f"Harvey HT master DF: {harvey_ht_master.shape}")
print(f"By subset: {harvey_ht_master['subset_label'].value_counts().to_dict()}")
print(f"By label: {harvey_ht_master['polyreactive_binary'].value_counts().to_dict()}")

Harvey HT master DF: (2000, 21)
By subset: {'harvey_ht_high': 1000, 'harvey_ht_low': 1000}
By label: {1: 1000, 0: 1000}


In [ ]:
# Build Harvey INDEXED PANEL (low-throughput) rows with continuous PSR scores
def build_harvey_lt_rows(df):
    rows = []
    for i, r in df.iterrows():
        biorep = r.get('Biorep average')
        # Binarize via median-of-indexed-panel for a reasonable binary label
        # but also keep continuous for Spearman
        rows.append({
            'source': 'harvey_2022',
            'ab_id': f"harvey_idx_{r.get('ID', i)}",
            'species': 'alpaca' if pd.isna(r.get('hmm_species')) else str(r.get('hmm_species')).lower(),
            'chain_type': 'H',
            'VH': r['VH_full'] if pd.notna(r.get('VH_full')) else None,
            'VL': None,
            'CDR-H1': r.get('CDR1_nogaps'),
            'CDR-H2': r.get('CDR2_nogaps'),
            'CDR-H3': r.get('CDR3_nogaps'),
            'v_gene': r.get('v_gene') if pd.notna(r.get('v_gene')) else None,
            'v_identity': float(r['v_identity']) if pd.notna(r.get('v_identity')) else None,
            'v_family': None,
            'j_gene': r.get('j_gene') if pd.notna(r.get('j_gene')) else None,
            'isotype': None,
            'format': 'VHH',
            'polyreactive_binary': None,  # will compute below
            'polyreactive_continuous': float(biorep) if pd.notna(biorep) else None,
            'assay_type': 'ELISA_PSR_biorep',
            'antigen_context': 'Sf9_insect_cell_lysate',
            'assay_family': 'harvey_PSR',
            'subset_label': 'harvey_indexed_panel',
        })
    return pd.DataFrame(rows)

harvey_lt_master = build_harvey_lt_rows(lt_df)

# Binarize: use median split within the indexed panel
lt_median = harvey_lt_master['polyreactive_continuous'].median()
harvey_lt_master['polyreactive_binary'] = (harvey_lt_master['polyreactive_continuous'] > lt_median).astype(int)

print(f"Harvey indexed panel master DF: {harvey_lt_master.shape}")
print(f"  Continuous PSR: median={lt_median:.2f}, range={harvey_lt_master['polyreactive_continuous'].min():.2f}"
      f" to {harvey_lt_master['polyreactive_continuous'].max():.2f}")
print(f"  Binary split at median: {harvey_lt_master['polyreactive_binary'].value_counts().to_dict()}")

# Drop any rows with None VH (can't embed)
harvey_lt_master = harvey_lt_master[harvey_lt_master['VH'].notna()].reset_index(drop=True)
print(f"  After dropping missing VH: {len(harvey_lt_master)}")

Harvey indexed panel master DF: (48, 21)
  Continuous PSR: median=16.77, range=1.77 to 99.19
  Binary split at median: {0: 24, 1: 24}
  After dropping missing VH: 48


In [ ]:
# Combine HT + LT into one Harvey master
harvey_master = pd.concat([harvey_ht_master, harvey_lt_master], ignore_index=True)

# Align columns to existing master (add missing, drop extra to match schema)
existing_cols = list(master_df_existing.columns)
for c in existing_cols:
    if c not in harvey_master.columns:
        harvey_master[c] = None
harvey_master_aligned = harvey_master[[c for c in existing_cols if c in harvey_master.columns] +
                                      [c for c in harvey_master.columns if c not in existing_cols]]

print(f"\nHarvey master (combined):")
print(f"  Shape: {harvey_master.shape}")
print(f"  By subset: {harvey_master['subset_label'].value_counts().to_dict()}")
print(f"  By polyreactive_binary: {harvey_master['polyreactive_binary'].value_counts().to_dict()}")
print(f"\nSchema alignment:")
print(f"  Existing master has {len(existing_cols)} columns")
print(f"  Harvey master has {len(harvey_master.columns)} columns")
print(f"  Missing from Harvey (will be None): {set(existing_cols) - set(harvey_master.columns)}")
print(f"  Extra in Harvey (preserved): {set(harvey_master.columns) - set(existing_cols)}")


Harvey master (combined):
  Shape: (2048, 52)
  By subset: {'harvey_ht_high': 1000, 'harvey_ht_low': 1000, 'harvey_indexed_panel': 48}
  By polyreactive_binary: {1: 1024, 0: 1024}

Schema alignment:
  Existing master has 38 columns
  Harvey master has 52 columns
  Missing from Harvey (will be None): set()
  Extra in Harvey (preserved): {'assay_family', 'v_gene', 'VL', 'ab_id', 'species', 'subset_label', 'format', 'VH', 'isotype', 'polyreactive_binary', 'j_gene', 'polyreactive_continuous', 'assay_type', 'antigen_context'}


## §5. AbLang embedding

~2,050 sequences. AbLang's heavy-chain model handles nanobodies (VHH) imperfectly but functionally — nanobody framework is structurally homologous to VH domains. Embeddings will be lower-quality than for conventional antibodies, but still capture signal.

In [ ]:
# Initialize AbLang heavy-chain model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print("Loading AbLang heavy chain model...")
heavy_model = ablang.pretrained('heavy', device=device)
heavy_model.freeze()
print("✓ AbLang heavy model loaded")

Using device: cuda
Loading AbLang heavy chain model...
✓ AbLang heavy model loaded


In [ ]:
# Embed in batches. AbLang returns per-residue embeddings; we mean-pool to get per-sequence 768-dim.
sequences = harvey_master['VH'].tolist()
batch_size = 64
embeddings_list = []

total = len(sequences)
print(f"Embedding {total} sequences in batches of {batch_size}...")

for i in range(0, total, batch_size):
    batch = sequences[i:i+batch_size]
    # mode='seqcoding' gives per-sequence 768-dim embedding (mean-pooled internally)
    batch_emb = heavy_model(batch, mode='seqcoding')
    embeddings_list.append(batch_emb)
    if (i // batch_size) % 5 == 0:
        pct = (i + len(batch)) / total * 100
        print(f"  {i + len(batch):>5} / {total}  ({pct:.1f}%)")

harvey_embeddings = np.vstack(embeddings_list)
print(f"\n✓ Embeddings computed: {harvey_embeddings.shape}")
print(f"  Mean: {harvey_embeddings.mean():.4f}  Std: {harvey_embeddings.std():.4f}")
assert harvey_embeddings.shape[0] == len(harvey_master), "Embedding count mismatch!"

Embedding 2048 sequences in batches of 64...
     64 / 2048  (3.1%)
    384 / 2048  (18.8%)
    704 / 2048  (34.4%)
   1024 / 2048  (50.0%)
   1344 / 2048  (65.6%)
   1664 / 2048  (81.2%)
   1984 / 2048  (96.9%)

✓ Embeddings computed: (2048, 768)
  Mean: 0.0011  Std: 0.4756


## §6. Merge into unified master

Append Harvey rows and embeddings to the existing 02b unified master. Final result is usable directly in Notebook 03c.

In [ ]:
existing_emb = np.load(UNIFIED_EMB_NPY)
print(f"Existing master: {master_df_existing.shape}")
print(f"Existing embeddings: {existing_emb.shape}")

assert existing_emb.shape[1] == harvey_embeddings.shape[1], (
    f"Dimension mismatch: {existing_emb.shape[1]} vs {harvey_embeddings.shape[1]}"
)

# Align Harvey columns to existing schema (critical for concat)
harvey_aligned = pd.DataFrame()
for col in master_df_existing.columns:
    if col in harvey_master.columns:
        harvey_aligned[col] = harvey_master[col].values
    else:
        harvey_aligned[col] = [None] * len(harvey_master)
# Preserve any Harvey-only metadata
for col in harvey_master.columns:
    if col not in harvey_aligned.columns:
        harvey_aligned[col] = harvey_master[col].values

# Extend existing to have the Harvey-only columns as None
existing_extended = master_df_existing.copy()
for col in harvey_aligned.columns:
    if col not in existing_extended.columns:
        existing_extended[col] = None

# Align column order
all_cols = list(existing_extended.columns)
harvey_aligned = harvey_aligned[all_cols]

# Concat
unified_master_with_harvey = pd.concat([existing_extended, harvey_aligned], ignore_index=True)
unified_embeddings_with_harvey = np.vstack([existing_emb, harvey_embeddings])

print(f"\nCombined unified master: {unified_master_with_harvey.shape}")
print(f"Combined unified embeddings: {unified_embeddings_with_harvey.shape}")

# Verify alignment
assert len(unified_master_with_harvey) == len(unified_embeddings_with_harvey), "Row/embedding count mismatch!"

print(f"\nSource distribution after Harvey:")
print(unified_master_with_harvey['source'].value_counts().to_string())

Existing master: (7700, 38)
Existing embeddings: (7700, 768)

Combined unified master: (9748, 52)
Combined unified embeddings: (9748, 768)

Source distribution after Harvey:
source
cov_abdab                  3000
harvey_2022                2048
oas_naive                  2000
thera_sabdab                994
boughter_2020_mouse         481
shehata_2019                400
boughter_2020_flu           379
jain_2017                   137
boughter_2020_nat_hiv       134
boughter_2020_gut_hiv        75
boughter_2020_nat_cntrl      50
boughter_2020_plos_hiv       50


## §7. Save outputs

In [ ]:
# Harvey-only artifacts
harvey_out_pkl = DRIVE_OUTPUT / f'harvey_processed_v{PROJECT_VERSION}.pkl'
harvey_out_emb = DRIVE_OUTPUT / f'harvey_embeddings_v{PROJECT_VERSION}.npy'
harvey_idx_pkl = DRIVE_OUTPUT / f'harvey_index_panel_v{PROJECT_VERSION}.pkl'

harvey_master.to_pickle(harvey_out_pkl)
np.save(harvey_out_emb, harvey_embeddings)
harvey_lt_master.to_pickle(harvey_idx_pkl)

# Extended unified master (for use in 03c)
unified_out_pkl = DRIVE_OUTPUT / f'unified_master_with_harvey_v{PROJECT_VERSION}.pkl'
unified_out_emb = DRIVE_OUTPUT / f'unified_embeddings_with_harvey_v{PROJECT_VERSION}.npy'

unified_master_with_harvey.to_pickle(unified_out_pkl)
np.save(unified_out_emb, unified_embeddings_with_harvey)

# Ingestion summary
summary = {
    'version': PROJECT_VERSION,
    'harvey_counts': {
        'high_pool_available': 71772,
        'low_pool_available': 69702,
        'high_sampled': 1000,
        'low_sampled': 1000,
        'indexed_panel_total': 48,
        'indexed_panel_usable': int(len(harvey_lt_master)),
        'random_seed': RANDOM_SEED,
    },
    'harvey_master_shape': list(harvey_master.shape),
    'harvey_embeddings_shape': list(harvey_embeddings.shape),
    'unified_master_with_harvey_shape': list(unified_master_with_harvey.shape),
    'unified_embeddings_with_harvey_shape': list(unified_embeddings_with_harvey.shape),
    'source_distribution_after': unified_master_with_harvey['source'].value_counts().to_dict(),
    'indexed_panel_continuous_stats': {
        'min': float(harvey_lt_master['polyreactive_continuous'].min()),
        'median': float(harvey_lt_master['polyreactive_continuous'].median()),
        'max': float(harvey_lt_master['polyreactive_continuous'].max()),
        'mean': float(harvey_lt_master['polyreactive_continuous'].mean()),
    },
}

with open(DRIVE_OUTPUT / f'ingestion_summary_v{PROJECT_VERSION}.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print("✓ All outputs saved:")
for path in [harvey_out_pkl, harvey_out_emb, harvey_idx_pkl, unified_out_pkl, unified_out_emb,
             DRIVE_OUTPUT / f'ingestion_summary_v{PROJECT_VERSION}.json']:
    print(f"  {path}  ({path.stat().st_size/1024/1024:.2f} MB)")

✓ All outputs saved:
  /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0/harvey_processed_v0.1.0.pkl  (0.58 MB)
  /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0/harvey_embeddings_v0.1.0.npy  (12.00 MB)
  /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0/harvey_index_panel_v0.1.0.pkl  (0.01 MB)
  /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0/unified_master_with_harvey_v0.1.0.pkl  (3.95 MB)
  /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0/unified_embeddings_with_harvey_v0.1.0.npy  (57.12 MB)
  /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0/ingestion_summary_v0.1.0.json  (0.00 MB)


## §8. Final checks

In [ ]:
checks = []
checks.append(('Harvey master has expected rows (~2048)',
               1900 <= len(harvey_master) <= 2100))
checks.append(('Harvey embeddings shape matches master',
               harvey_embeddings.shape[0] == len(harvey_master)))
checks.append(('Harvey embeddings are 768-dim',
               harvey_embeddings.shape[1] == 768))
checks.append(('Harvey has balanced HIGH/LOW binary labels',
               abs(harvey_master['polyreactive_binary'].sum() - len(harvey_master)/2) < 100))
checks.append(('Indexed panel has continuous scores',
               harvey_lt_master['polyreactive_continuous'].notna().all()))
checks.append(('Unified master grew by ~2048 rows',
               1900 <= (len(unified_master_with_harvey) - len(master_df_existing)) <= 2100))
checks.append(('Unified master/embedding counts match',
               len(unified_master_with_harvey) == len(unified_embeddings_with_harvey)))
checks.append(('No NaN VH sequences in Harvey master',
               harvey_master['VH'].notna().all()))

print(f"\n{'='*60}")
print("FINAL CHECKS")
print(f"{'='*60}")
for name, passed in checks:
    print(f"  {'✓ PASS' if passed else '✗ FAIL'}  {name}")

n_pass = sum(1 for _, p in checks if p)
print(f"\n{n_pass}/{len(checks)} checks passed")

if n_pass == len(checks):
    print("🟢 All checks passed. Ready for Notebook 03c (J/JS/JSB on Harvey).")
else:
    print("🔴 Some checks failed. Investigate before proceeding.")
print(f"{'='*60}")

print(f"\n📦 Artifacts for Notebook 03c:")
print(f"  Master DF:    {unified_out_pkl}")
print(f"  Embeddings:   {unified_out_emb}")
print(f"  Index panel:  {harvey_idx_pkl}")


FINAL CHECKS
  ✓ PASS  Harvey master has expected rows (~2048)
  ✓ PASS  Harvey embeddings shape matches master
  ✓ PASS  Harvey embeddings are 768-dim
  ✓ PASS  Harvey has balanced HIGH/LOW binary labels
  ✓ PASS  Indexed panel has continuous scores
  ✓ PASS  Unified master grew by ~2048 rows
  ✓ PASS  Unified master/embedding counts match
  ✓ PASS  No NaN VH sequences in Harvey master

8/8 checks passed
🟢 All checks passed. Ready for Notebook 03c (J/JS/JSB on Harvey).

📦 Artifacts for Notebook 03c:
  Master DF:    /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0/unified_master_with_harvey_v0.1.0.pkl
  Embeddings:   /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0/unified_embeddings_with_harvey_v0.1.0.npy
  Index panel:  /content/drive/MyDrive/PolyAtlas/PolyAtlas_02c_harvey_ingestion_v0.1.0/harvey_index_panel_v0.1.0.pkl


## What's next

With Harvey ingested, **Notebook 03c** will run the proper J/JS/JSB experiment:

| Condition | Training | Assay diversity | Test |
|---|---|---|---|
| **J** | Jain only | 1 (Adimab PSR) | Harvey |
| **JS** | Jain + Shehata | 1 (Adimab PSR, more data) | Harvey |
| **JSB** | Jain + Shehata + Boughter | 2 (Adimab PSR + Boughter ELISA) | Harvey |

Harvey is entirely out-of-distribution from all three training conditions. The relative performance across J → JS → JSB isolates the **assay-diversity effect** from the **more-data effect** (J → JS adds data without assay diversity).

Additional analysis in 03c:
- Binary AUC on the 2,000-nanobody HT Harvey test (high vs low pool)
- Spearman correlation against the 48 indexed nanobody continuous PSR scores
- Per-condition bootstrap 95% CIs
- Caveat documentation on nanobody domain shift

---
*End of PolyAtlas_02c_harvey_ingestion v0.1.0*